# Восстановление пунктуации (мультимодальное, late fusion) — корпус M-AILABS

Модель преобразует **текст без пунктуации → текст с пунктуацией**, опираясь *одновременно* на текст и на акустические признаки из звука (паузы, длительности, темп, **F0**, энергия). Восстанавливаются **запятые, точки, многоточия, вопросительные и восклицательные знаки**, а также **абзацы (красные строки)** и **капитализация**.

## Почему M-AILABS, а не FLEURS
FLEURS — это начитка энциклопедических статей: почти одни повествовательные предложения, поэтому `?` `!` `…` там почти нет, а абзацев нет вовсе. **M-AILABS** — русские аудиокниги (LibriVox/Gutenberg), художественный текст с **полной книжной пунктуацией** (диалоги дают вопросы и восклицания) и **реальными абзацами**. Поэтому классы `QUESTION/EXCLAM/ELLIPSIS` и голова `PARA` наконец получают обучающий сигнал.

## Что исправлено в этой версии
1. **Метрика macro-F1** теперь усредняется только по классам, реально присутствующим в данных (`support > 0`). Раньше пустые классы (`QUESTION` с support=0) механически занижали F1 до ~0.35, создавая ложное впечатление плохого качества.
2. **Focal Loss** (`loss_type="focal"`) против дисбаланса класса `O` — обычно лучший рычаг, когда «accuracy высокая, а знаки не ставятся».
3. **Автоподбор весов классов** по реальной частоте в train (`auto_class_weights=True`).
4. **Извлечение абзацев** из книжного текста (`parse_document`) — голова `PARA` обучается.
5. `pretty_report` предупреждает, что **accuracy на этой задаче смотреть нельзя**.

## Архитектуры
BiLSTM (baseline) · Transformer с нуля (baseline-трансформер) · RuBERT-base и rubert-tiny2 (предобученные). Все двухпоточные: текст ⊕ акустика → 3 головы.

## 0. Зависимости
Раскомментируйте при первом запуске.

In [5]:
# !pip install -r requirements.txt
# Forced aligner (акустика: паузы/F0) — нужен для мультимодального режима:
# !pip install git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git

import os
os.environ.setdefault("DATASETS_AUDIO_BACKEND", "soundfile")  # Windows: чтение аудио

'soundfile'

## 1. Импорт модулей

In [13]:
import numpy as np
import torch
from functools import partial
from torch.utils.data import DataLoader

from modules.models.__init__ import *
from modules.models.heads import *
from modules.models.lstm_model import *
from modules.models.pretrained_model import *
from modules.models.transformer_model import *
from modules.config import *
from modules.data import *
from modules.dataset import *
from modules.evaluate import *
from modules.inference import *
from modules.tokenizer import *
from modules.train import *
    # get_config,
    # build_examples, WordVocab,
    # BaselineDataset, PretrainedDataset, baseline_collate, pretrained_collate,
    # build_model, load_hf_tokenizer,
    # train_model, set_seed,
    # evaluate, pretty_report,
    # PunctuationRestorer, STTPunctuationPipeline,
    # PRETRAINED_PRESETS, PUNCT_LABELS, PARA_LABELS, CAP_LABELS, ACOUSTIC_FEATURES,
# )

cfg = get_config()
set_seed(cfg.train.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.train.device = DEVICE
print("device:", DEVICE)
print("пунктуация:", PUNCT_LABELS, "| абзац:", PARA_LABELS, "| капитализация:", CAP_LABELS)
print("акустика:", ACOUSTIC_FEATURES)
print("loss:", cfg.train.loss_type, "| автовеса классов:", cfg.train.auto_class_weights)

device: cpu
пунктуация: ['O', 'COMMA', 'PERIOD', 'QUESTION', 'EXCLAM', 'ELLIPSIS'] | абзац: ['NO_PARA', 'PARA'] | капитализация: ['LOWER', 'CAP', 'UPPER']
акустика: ['pause_before', 'pause_after', 'word_duration', 'speech_rate', 'f0_end_median', 'f0_end_slope', 'energy_end']
loss: focal | автовеса классов: True


## 2. Данные: M-AILABS (русские аудиокниги)

`build_examples(source="mailabs", ...)` грузит M-AILABS, делит на train/val/test сам (датасет идёт одним сплитом), извлекает метки трёх голов и абзацы, и — если установлен forced-aligner — акустику.

* `use_alignment=True` + установленный aligner → мультимодальный режим (паузы/F0).
* `use_alignment=False` → text-only (акустика = нули), быстрый прогон.
* `mix_fleurs=True` → подмешать FLEURS для разнообразия дикторов/тем.
* Если корпус/сеть недоступны → демо-примеры (с абзацами), чтобы ноутбук исполнялся.

При первом запуске M-AILABS скачается (несколько ГБ). Подберите `LIMIT_*` под железо.

In [14]:
USE_ALIGNMENT = True   # False -> быстрый text-only прогон
MIX_FLEURS    = False  # True -> добавить FLEURS к M-AILABS
LIMIT_TRAIN   = 4000   # None = весь train; аудиокниги крупные, начните умеренно
LIMIT_VAL     = 800

train_examples = build_examples(cfg.data, split="train",      limit=LIMIT_TRAIN,
                                use_alignment=USE_ALIGNMENT, source="mailabs", mix_fleurs=MIX_FLEURS)
val_examples   = build_examples(cfg.data, split="validation", limit=LIMIT_VAL,
                                use_alignment=USE_ALIGNMENT, source="mailabs", mix_fleurs=MIX_FLEURS)

print(f"train: {len(train_examples)} | val: {len(val_examples)}")
ex = train_examples[0]
print("слова:", ex.words[:12])
print("есть акустика:", ex.has_acoustic, "| форма:", ex.acoustic.shape)

# распределение классов пунктуации — убедимся, что ? ! … теперь присутствуют
from collections import Counter
c = Counter(x for e in train_examples for x in e.punct_ids)
print("распределение пунктуации:", {PUNCT_LABELS[k]: c[k] for k in sorted(c)})
par = Counter(x for e in train_examples for x in e.para_ids)
print("абзацы (NO_PARA/PARA):", {PARA_LABELS[k]: par[k] for k in sorted(par)})

[build_examples] forced-aligner не установлен — text-only режим.
[load_mailabs] не удалось загрузить M-AILABS ни с одного зеркала (последняя ошибка: Dataset 'mailabs/ru_RU' doesn't exist on the Hub or cannot be accessed.).
[build_examples] корпус недоступен — возвращаю демо-примеры (text-only).
[build_examples] forced-aligner не установлен — text-only режим.
[load_mailabs] не удалось загрузить M-AILABS ни с одного зеркала (последняя ошибка: Dataset 'mailabs/ru_RU' doesn't exist on the Hub or cannot be accessed.).
[build_examples] корпус недоступен — возвращаю демо-примеры (text-only).
train: 2 | val: 2
слова: ['привет', 'как', 'дела', 'я', 'давно', 'тебя', 'не', 'видел', 'сегодня', 'хорошая', 'погода', 'может']
есть акустика: False | форма: (15, 7)
распределение пунктуации: {'O': 21, 'COMMA': 3, 'PERIOD': 3, 'QUESTION': 2, 'EXCLAM': 2, 'ELLIPSIS': 1}
абзацы (NO_PARA/PARA): {'NO_PARA': 30, 'PARA': 2}


### (опц.) Пересчёт нормализации акустики на train
Если включена акустика, грубые `ACOUSTIC_NORM` лучше заменить реальными mean/std.

In [15]:
# from modules.data import compute_acoustic_stats
# import pprint; pprint.pprint(compute_acoustic_stats(train_examples))

---
## 3. Baseline №1 — BiLSTM
Обучение использует Focal Loss и автовеса классов (передаём `train_examples`).

In [16]:
vocab = WordVocab.build(train_examples, min_freq=1, max_size=50000)
print("словарь:", len(vocab))

collate = partial(baseline_collate, pad_id=vocab.pad_id)
train_loader = DataLoader(BaselineDataset(train_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(BaselineDataset(val_examples, vocab, cfg.train.max_len),
                          batch_size=cfg.train.batch_size, shuffle=False, collate_fn=collate)

lstm = build_model("lstm", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in lstm.parameters()))

словарь: 32
параметров: 2648971


In [17]:
cfg.train.epochs = 8
cfg.train.lr = 1e-3
lstm = train_model(lstm, train_loader, cfg.train, val_loader=val_loader,
                   eval_fn=evaluate, train_examples=train_examples)  # <-- train_examples для автовесов
lstm_metrics = evaluate(lstm, val_loader, torch.device(DEVICE))
print(pretty_report(lstm_metrics))

[train] loss=focal (gamma=2.0), auto_class_weights=True
[epoch 1/8] train_loss=0.9088  val_punct_F1=0.0000
[epoch 2/8] train_loss=0.8140  val_punct_F1=0.2000
[epoch 3/8] train_loss=0.7473  val_punct_F1=0.2000
[epoch 4/8] train_loss=0.6721  val_punct_F1=0.2000
[epoch 5/8] train_loss=0.6350  val_punct_F1=0.2000
[epoch 6/8] train_loss=0.5767  val_punct_F1=0.2000
[epoch 7/8] train_loss=0.5286  val_punct_F1=0.2000
[epoch 8/8] train_loss=0.5265  val_punct_F1=0.2000
[train] загружены лучшие веса (val_punct_F1=0.2000).
ВНИМАНИЕ: accuracy на этой задаче обманчива — класс «нет знака» (O)
преобладает (~80-90%), поэтому ориентируйтесь на recall/F1 по классам
знаков и на macro-F1 (он считается только по присутствующим классам).

=== PUNCT (macro-F1=0.200 по 5 присутствующим классам, acc=0.688) ===
class              P       R      F1   support
O              0.677   1.000   0.808        21
COMMA          0.000   0.000   0.000         3
PERIOD         0.000   0.000   0.000         3
QUESTION       0

---
## 4. Baseline №2 — Transformer с нуля

In [18]:
transformer = build_model("transformer", vocab_size=len(vocab), pad_id=vocab.pad_id, use_acoustic=True)
print("параметров:", sum(p.numel() for p in transformer.parameters()))

cfg.train.epochs = 10   # трансформеру с нуля нужно больше эпох/данных
cfg.train.lr = 3e-4
transformer = train_model(transformer, train_loader, cfg.train, val_loader=val_loader,
                          eval_fn=evaluate, train_examples=train_examples)
tr_metrics = evaluate(transformer, val_loader, torch.device(DEVICE))
print(pretty_report(tr_metrics))

параметров: 3175563
[train] loss=focal (gamma=2.0), auto_class_weights=True
[epoch 1/10] train_loss=0.9776  val_punct_F1=0.0421
[epoch 2/10] train_loss=0.9728  val_punct_F1=0.3905

c:\Users\Roman\Documents\Projects\STT_russian_lang\venv\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(



[epoch 3/10] train_loss=0.5905  val_punct_F1=0.5933
[epoch 4/10] train_loss=0.3815  val_punct_F1=0.8667
[epoch 5/10] train_loss=0.2835  val_punct_F1=1.0000
[epoch 6/10] train_loss=0.1883  val_punct_F1=1.0000
[epoch 7/10] train_loss=0.1458  val_punct_F1=1.0000
[epoch 8/10] train_loss=0.1055  val_punct_F1=1.0000
[epoch 9/10] train_loss=0.0962  val_punct_F1=1.0000
[epoch 10/10] train_loss=0.0993  val_punct_F1=1.0000
[train] загружены лучшие веса (val_punct_F1=1.0000).
ВНИМАНИЕ: accuracy на этой задаче обманчива — класс «нет знака» (O)
преобладает (~80-90%), поэтому ориентируйтесь на recall/F1 по классам
знаков и на macro-F1 (он считается только по присутствующим классам).

=== PUNCT (macro-F1=1.000 по 5 присутствующим классам, acc=1.000) ===
class              P       R      F1   support
O              1.000   1.000   1.000        21
COMMA          1.000   1.000   1.000         3
PERIOD         1.000   1.000   1.000         3
QUESTION       1.000   1.000   1.000         2
EXCLAM         

---
## 5. Предобученные — RuBERT-base / rubert-tiny2

In [19]:
PRESET = "rubert-base"   # или "rubert-tiny2" (легче/быстрее)
model_name = PRETRAINED_PRESETS[PRESET]
print("модель:", model_name)

hf_tok = load_hf_tokenizer(model_name)
ptr_collate = partial(pretrained_collate, pad_id=hf_tok.pad_token_id or 0)
ptr_train_loader = DataLoader(PretrainedDataset(train_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=True, collate_fn=ptr_collate)
ptr_val_loader   = DataLoader(PretrainedDataset(val_examples, hf_tok, cfg.train.max_len),
                              batch_size=cfg.train.batch_size, shuffle=False, collate_fn=ptr_collate)

pretrained = build_model("pretrained", model_name=model_name, use_acoustic=True)

модель: DeepPavlov/rubert-base-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
cfg.train.epochs = 3   # предобученным хватает меньше эпох
pretrained = train_model(pretrained, ptr_train_loader, cfg.train, val_loader=ptr_val_loader,
                         is_pretrained=True, eval_fn=evaluate, train_examples=train_examples)
ptr_metrics = evaluate(pretrained, ptr_val_loader, torch.device(DEVICE))
print(pretty_report(ptr_metrics))

[train] loss=focal (gamma=2.0), auto_class_weights=True
[epoch 1/3] train_loss=0.9404  val_punct_F1=0.0000
[epoch 2/3] train_loss=0.6940  val_punct_F1=0.0000
[epoch 3/3] train_loss=0.6067  val_punct_F1=0.0000
[train] загружены лучшие веса (val_punct_F1=0.0000).
ВНИМАНИЕ: accuracy на этой задаче обманчива — класс «нет знака» (O)
преобладает (~80-90%), поэтому ориентируйтесь на recall/F1 по классам
знаков и на macro-F1 (он считается только по присутствующим классам).

=== PUNCT (macro-F1=0.000 по 5 присутствующим классам, acc=0.656) ===
class              P       R      F1   support
O              0.656   1.000   0.792        21
COMMA          0.000   0.000   0.000         3
PERIOD         0.000   0.000   0.000         3
QUESTION       0.000   0.000   0.000         2
EXCLAM         0.000   0.000   0.000         2
ELLIPSIS       0.000   0.000   0.000         1

=== PARA (macro-F1=0.000 по 1 присутствующим классам, acc=0.938) ===
class              P       R      F1   support
NO_PARA      

### Сравнение моделей

In [21]:
import pandas as pd
rows = [{"модель": n,
         "punct F1": round(m.get("punct_f1_macro", 0), 3),
         "para F1":  round(m.get("para_f1_macro", 0), 3),
         "cap F1":   round(m.get("cap_f1_macro", 0), 3)}
        for n, m in [("BiLSTM", lstm_metrics), ("Transformer", tr_metrics), (PRESET, ptr_metrics)]]
pd.DataFrame(rows)

,модель,punct F1,para F1,cap F1
0,BiLSTM,0.2,0.333,0.429
1,Transformer,1.0,1.000,0.571
2,rubert-base,0.0,0.000,0.000


---
## 6. Инференс

In [25]:
restorer_lstm = PunctuationRestorer(lstm, kind="lstm", vocab=vocab, device=DEVICE)
# предобученная: PunctuationRestorer(pretrained, kind="pretrained", hf_tokenizer=hf_tok, device=DEVICE)

print(restorer_lstm.restore("привет как дела я давно тебя не видел"))
print(restorer_lstm.restore("что это было невероятно я не ожидал такого поворота событий"))

Привет Как дела я давно тебя не видел
Что Это Было Невероятно я не ожидал такого поворота

событий


In [26]:
restorer_transformer = PunctuationRestorer(transformer, kind="transformer", vocab=vocab, device=DEVICE)
# предобученная: PunctuationRestorer(pretrained, kind="pretrained", hf_tokenizer=hf_tok, device=DEVICE)

print(restorer_transformer.restore("привет как дела я давно тебя не видел"))
print(restorer_transformer.restore("что это было невероятно я не ожидал такого поворота событий"))

Привет, как дела? Я давно тебя не видел
Что это было? невероятно я не ожидал такого поворота событий.


## 7. Встраивание в SpeechToText-пайплайн
Whisper отдаёт слова + тайм-коды; из них считаются те же акустические признаки, что при обучении (паузы → границы, F0 → `?`/`!`).

In [27]:
pipeline = STTPunctuationPipeline(restorer_lstm)

words = "что это было невероятно я не ожидал такого".split()
t, word_ts = 0.0, []
for w in words:
    word_ts.append({"word": w, "start": round(t,2), "end": round(t+0.3,2)})
    t += 0.3 + (0.7 if w in ("было","невероятно","такого") else 0.05)

print("С паузами:", pipeline({"words": words, "word_timestamps": word_ts, "audio": None}))
print("Текст    :", pipeline({"text": " ".join(words)}))

С паузами: Что это было? невероятно! я не ожидал такого
Текст    : Что Это Было Невероятно я не ожидал такого


In [28]:
pipeline = STTPunctuationPipeline(restorer_transformer)

words = "что это было невероятно я не ожидал такого".split()
t, word_ts = 0.0, []
for w in words:
    word_ts.append({"word": w, "start": round(t,2), "end": round(t+0.3,2)})
    t += 0.3 + (0.7 if w in ("было","невероятно","такого") else 0.05)

print("С паузами:", pipeline({"words": words, "word_timestamps": word_ts, "audio": None}))
print("Текст    :", pipeline({"text": " ".join(words)}))

С паузами: Что это было? невероятно я не ожидал такого
Текст    : Что это было? невероятно я не ожидал такого


### Реальный Whisper
```python
import whisper, soundfile as sf
asr = whisper.load_model("large-v3")
res = asr.transcribe("audio.wav", language="ru", word_timestamps=True)
words, word_ts = [], []
for seg in res["segments"]:
    for w in seg["words"]:
        tok = w["word"].strip(); words.append(tok)
        word_ts.append({"word": tok, "start": w["start"], "end": w["end"]})
audio, sr = sf.read("audio.wav")
final = STTPunctuationPipeline(restorer)({"words": words, "word_timestamps": word_ts, "audio": audio}, sr=sr)
```

## 8. Настройка против дисбаланса (если знаки всё ещё редки)
Все рычаги в `cfg.train` (модуль `config.py`):
```python
cfg.train.loss_type = "focal"      # "ce" — обычный взвешенный CrossEntropy
cfg.train.focal_gamma = 2.0        # больше -> сильнее фокус на редких знаках (попробуйте 3.0)
cfg.train.auto_class_weights = True  # автовеса по частоте в train
```
И помните: **смотрите recall по знакам и macro-F1, а не accuracy.**

## 9. Сохранение
```python
torch.save(lstm.state_dict(), "lstm_punct.pt"); vocab.save("vocab.json")
torch.save(pretrained.state_dict(), "rubert_punct.pt")
```